In [1]:
import polars as pl

auth = pl.scan_parquet("filtered_auth.parquet")

# compute split cutoffs from AUTH time (same units)
times = auth.select(pl.col("time").cast(pl.Int64)).collect()
q70 = int(times["time"].quantile(0.70))
q85 = int(times["time"].quantile(0.85))

train_auth = auth.filter(pl.col("time") <= q70)
val_auth   = auth.filter((pl.col("time") > q70) & (pl.col("time") <= q85))
test_auth  = auth.filter(pl.col("time") > q85)

def uniq_users(ldf: pl.LazyFrame) -> int:
    return (
        ldf.filter(pl.col("src_user").str.starts_with("U"))
           .select(pl.col("src_user").alias("user"))
           .unique()
           .collect()
           .height
    )

full_users  = uniq_users(auth)
train_users = uniq_users(train_auth)
val_users   = uniq_users(val_auth)
test_users  = uniq_users(test_auth)

print("Unique users in FULL auth:", full_users)
print("Unique users in TRAIN window:", train_users)
print("Unique users in VAL window:", val_users)
print("Unique users in TEST window:", test_users)

Unique users in FULL auth: 25951
Unique users in TRAIN window: 23803
Unique users in VAL window: 15880
Unique users in TEST window: 17607


In [2]:
import polars as pl

auth = pl.scan_parquet("filtered_auth.parquet")
auth_users_all = (
    auth.filter(pl.col("src_user").str.starts_with("U"))
        .select(pl.col("src_user").alias("user"))
        .unique()
        .collect()
)

red = pl.scan_csv(
    "data_windows/redteam.txt.gz",
    has_header=False,
    new_columns=["time", "user", "src_computer", "dst_computer"],
).select("user").unique().collect()

overlap_all = auth_users_all.join(red, on="user", how="inner").height
print("Overlap (FULL auth users vs redteam users):", overlap_all)
print("Auth users:", auth_users_all.height)
print("Redteam users:", red.height)

Overlap (FULL auth users vs redteam users): 104
Auth users: 25951
Redteam users: 104


In [3]:
def overlap_with_redteam(ldf: pl.LazyFrame) -> int:
    split_users = (
        ldf.filter(pl.col("src_user").str.starts_with("U"))
           .select(pl.col("src_user").alias("user"))
           .unique()
           .collect()
    )
    return split_users.join(red, on="user", how="inner").height

print("Overlap in TRAIN window:", overlap_with_redteam(train_auth))
print("Overlap in VAL window:", overlap_with_redteam(val_auth))
print("Overlap in TEST window:", overlap_with_redteam(test_auth))

Overlap in TRAIN window: 104
Overlap in VAL window: 93
Overlap in TEST window: 96


In [4]:
# ============================
# A) UPDATED FEATURE ENGINEERING (Polars)
# ============================

import polars as pl

def norm_user(expr: pl.Expr) -> pl.Expr:
    return (
        expr.cast(pl.Utf8)
            .str.strip_chars()
            .str.split("@").list.first()
            .str.split("\\").list.last()
    )

def user_features(ldf: pl.LazyFrame) -> pl.LazyFrame:
    # keep only real users
    ldf = (
        ldf.filter(pl.col("src_user").str.starts_with("U"))
           .with_columns(norm_user(pl.col("src_user")).alias("user"))
           .with_columns([
               pl.col("time").cast(pl.Int64),
               ((pl.col("time") // 3600) % 24).cast(pl.Int32).alias("hour"),
               ((pl.col("time") // 86400) % 7).cast(pl.Int32).alias("dow"),
           ])
           .with_columns([
               (pl.col("dow") >= 5).cast(pl.Int8).alias("is_weekend"),
               (pl.col("hour").is_between(0, 5)).cast(pl.Int8).alias("is_night"),
               (pl.col("status") == "Failure").cast(pl.Int8).alias("is_fail"),
           ])
    )

    # core aggregates
    core = (
        ldf.group_by("user")
           .agg([
               pl.len().alias("total_logins"),
               pl.col("is_fail").sum().alias("failed_logins"),
               pl.col("dst_computer").n_unique().alias("unique_destinations"),
               pl.col("src_computer").n_unique().alias("unique_src_computers"),
               pl.col("is_fail").mean().alias("fail_rate"),
               pl.col("is_night").mean().alias("night_rate"),
               pl.col("is_weekend").mean().alias("weekend_rate"),
               pl.col("hour").n_unique().alias("unique_hours"),
           ])
           .with_columns([
               (pl.col("failed_logins") / (pl.col("total_logins") + 1e-6)).alias("fail_rate_2"),
               (pl.col("unique_destinations") / (pl.col("total_logins") + 1e-6)).alias("dst_per_login"),
               (pl.col("unique_src_computers") / (pl.col("total_logins") + 1e-6)).alias("src_per_login"),
           ])
    )

    # burstiness: inter-arrival time stats per user
    gaps = (
        ldf.select(["user", "time"])
           .sort(["user", "time"])
           .with_columns([
               (pl.col("time") - pl.col("time").shift(1)).over("user").alias("dt")
           ])
           .group_by("user")
           .agg([
               pl.col("dt").drop_nulls().median().alias("median_dt"),
               pl.col("dt").drop_nulls().mean().alias("mean_dt"),
               pl.col("dt").drop_nulls().std().alias("std_dt"),
               pl.col("dt").drop_nulls().quantile(0.90).alias("p90_dt"),
           ])
    )

    # destination concentration features
    dest_share = (
        ldf.group_by(["user", "dst_computer"])
           .agg(pl.len().alias("cnt"))
           .group_by("user")
           .agg([
               pl.col("cnt").max().alias("top_dst_cnt"),
               pl.col("cnt").sum().alias("dst_total_cnt"),
               pl.col("cnt").mean().alias("dst_mean_cnt"),
               pl.col("cnt").std().alias("dst_std_cnt"),
           ])
           .with_columns([
               (pl.col("top_dst_cnt") / (pl.col("dst_total_cnt") + 1e-6)).alias("top_dst_share")
           ])
    )

    

    # logon_type diversity + concentration
    logon_div = (
        ldf.group_by(["user", "logon_type"])
           .agg(pl.len().alias("lt_cnt"))
           .group_by("user")
           .agg([
               pl.col("logon_type").n_unique().alias("unique_logon_types"),
               pl.col("lt_cnt").max().alias("top_logon_cnt"),
               pl.col("lt_cnt").sum().alias("logon_total_cnt"),
           ])
           .with_columns([
               (pl.col("top_logon_cnt") / (pl.col("logon_total_cnt") + 1e-6)).alias("top_logon_share")
           ])
    )

    feat = (
        core.join(gaps, on="user", how="left")
            .join(dest_share, on="user", how="left")
            .join(logon_div, on="user", how="left")
            .with_columns(pl.all().exclude("user").fill_null(0))
    )
    return feat

# Build split cutoffs from auth time (same units as your parquet)
auth = pl.scan_parquet("filtered_auth.parquet")
times = auth.select(pl.col("time").cast(pl.Int64)).collect()
q70 = int(times["time"].quantile(0.70))
q85 = int(times["time"].quantile(0.85))

train_auth = auth.filter(pl.col("time") <= q70)
val_auth   = auth.filter((pl.col("time") > q70) & (pl.col("time") <= q85))
test_auth  = auth.filter(pl.col("time") > q85)

train_feat = user_features(train_auth)
val_feat   = user_features(val_auth)
test_feat  = user_features(test_auth)

# (Optional but recommended) deviation features using TRAIN baselines only
train_df = train_feat.collect()
val_df   = val_feat.collect()
test_df  = test_feat.collect()

base_feature_cols = [c for c in train_df.columns if c != "user"]

baseline = (
    train_df.select(["user"] + base_feature_cols)
            .group_by("user")
            .agg([pl.col(c).mean().alias(f"base_{c}") for c in base_feature_cols])
)


# def add_deviation(df: pl.DataFrame) -> pl.DataFrame:
#     out = df.join(baseline, on="user", how="left")
#     for c in base_feature_cols:
#         out = out.with_columns([
#             (pl.col(c) - pl.col(f"base_{c}")).fill_null(0).alias(f"diff_{c}"),
#             (pl.col(c) / (pl.col(f"base_{c}") + 1e-6)).fill_null(0).alias(f"ratio_{c}"),
#         ])
#     return out.drop([f"base_{c}" for c in base_feature_cols])

# train_df2 = add_deviation(train_df)
# val_df2   = add_deviation(val_df)
# test_df2  = add_deviation(test_df)

# # rebuild feature column list safely (numeric only)
# non_feature_cols = {"user", "label", "compromise_events", "src_user"}
# feature_cols2 = [
#     c for c, dtype in zip(train_df2.columns, train_df2.dtypes)
#     if c not in non_feature_cols and str(dtype) in ("Float64","Float32","Int64","Int32","Int16","Int8")
# ]

# print("num features:", len(feature_cols2))
# print(feature_cols2[:15])



# # ============================
# # B) AUTOENCODER MODEL CHECKING CODE (NO LABELS REQUIRED)
# # - trains on TRAIN
# # - selects best config by lowest VAL reconstruction loss
# # - prints test reconstruction ranking stats
# # ============================

# import numpy as np
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from sklearn.preprocessing import RobustScaler
# from torch.utils.data import Dataset, DataLoader

# X_train = train_df2.select(feature_cols2).to_numpy()
# X_val   = val_df2.select(feature_cols2).to_numpy()
# X_test  = test_df2.select(feature_cols2).to_numpy()

# scaler = RobustScaler()
# X_train_s = scaler.fit_transform(X_train)
# X_val_s   = scaler.transform(X_val)
# X_test_s  = scaler.transform(X_test)

# input_dim = X_train_s.shape[1]
# print("input_dim:", input_dim)

# class NumpyDataset(Dataset):
#     def __init__(self, X):
#         self.X = torch.tensor(X, dtype=torch.float32)
#     def __len__(self):
#         return self.X.shape[0]
#     def __getitem__(self, idx):
#         return self.X[idx]

# device = "cuda" if torch.cuda.is_available() else "cpu"
# print("device:", device)

# train_loader = DataLoader(NumpyDataset(X_train_s), batch_size=4096, shuffle=True, num_workers=0)
# val_tensor   = torch.tensor(X_val_s, dtype=torch.float32)
# test_tensor  = torch.tensor(X_test_s, dtype=torch.float32)

# class AutoEncoder(nn.Module):
#     def __init__(self, input_dim, hidden_dim=128, latent_dim=16, dropout=0.1):
#         super().__init__()
#         self.encoder = nn.Sequential(
#             nn.Linear(input_dim, hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(hidden_dim, latent_dim),
#             nn.ReLU(),
#         )
#         self.decoder = nn.Sequential(
#             nn.Linear(latent_dim, hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(hidden_dim, input_dim),
#         )

#     def forward(self, x):
#         return self.decoder(self.encoder(x))

# def train_ae(train_loader, val_tensor, input_dim, hidden_dim, latent_dim, dropout, lr, weight_decay, epochs=40, patience=5):
#     model = AutoEncoder(input_dim, hidden_dim, latent_dim, dropout).to(device)
#     opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
#     loss_fn = nn.MSELoss()

#     best_val = float("inf")
#     best_state = None
#     bad = 0

#     for _ in range(epochs):
#         model.train()
#         for xb in train_loader:
#             xb = xb.to(device)
#             opt.zero_grad(set_to_none=True)
#             loss = loss_fn(model(xb), xb)
#             loss.backward()
#             opt.step()

#         model.eval()
#         with torch.no_grad():
#             xv = val_tensor.to(device)
#             val_loss = loss_fn(model(xv), xv).item()

#         if val_loss < best_val - 1e-6:
#             best_val = val_loss
#             best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
#             bad = 0
#         else:
#             bad += 1
#             if bad >= patience:
#                 break

#     if best_state is not None:
#         model.load_state_dict(best_state)
#     return model, best_val

# def recon_scores_mae(model, X_tensor):
#     model.eval()
#     with torch.no_grad():
#         x = X_tensor.to(device)
#         r = model(x)
#         return (r - x).abs().mean(dim=1).detach().cpu().numpy()

# # sweep (unsupervised selection)
# search_space = {
#     "hidden_dim": [64, 128, 256],
#     "latent_dim": [8, 16, 32],
#     "dropout":    [0.0, 0.1, 0.2],
#     "lr":         [1e-3, 3e-4],
#     "weight_decay":[0.0, 1e-4],
# }

# best_model = None
# best_cfg = None
# best_val_loss = float("inf")

# total = (
#     len(search_space["hidden_dim"]) *
#     len(search_space["latent_dim"]) *
#     len(search_space["dropout"]) *
#     len(search_space["lr"]) *
#     len(search_space["weight_decay"])
# )
# trial = 0

# for hd in search_space["hidden_dim"]:
#     for ld in search_space["latent_dim"]:
#         for dr in search_space["dropout"]:
#             for lr in search_space["lr"]:
#                 for wd in search_space["weight_decay"]:
#                     trial += 1
#                     model, vloss = train_ae(train_loader, val_tensor, input_dim, hd, ld, dr, lr, wd)
#                     if vloss < best_val_loss:
#                         best_val_loss = vloss
#                         best_model = model
#                         best_cfg = {"hidden_dim": hd, "latent_dim": ld, "dropout": dr, "lr": lr, "weight_decay": wd}
#                         print(f"[{trial}/{total}] NEW BEST val_mse={best_val_loss:.6f} cfg={best_cfg}")
#                     elif trial % 10 == 0:
#                         print(f"[{trial}/{total}] current best val_mse={best_val_loss:.6f}")

# print("\nFINAL BEST AE CONFIG:", best_cfg)
# print("FINAL BEST VAL MSE:", best_val_loss)

# # show top anomalies in TEST by reconstruction error
# test_scores = recon_scores_mae(best_model, test_tensor)
# top_idx = np.argsort(test_scores)[::-1][:20]
# print("\nTop 20 test anomalies (by AE recon MAE):")
# print(top_idx)
# print(test_scores[top_idx])

In [5]:
import polars as pl

# choose which columns should NOT get ratio features
# dt stats and rates are the usual culprits
NO_RATIO_PREFIXES = ("median_dt", "mean_dt", "std_dt", "p90_dt")
NO_RATIO_EXACT = {"fail_rate", "fail_rate_2", "night_rate", "weekend_rate"}

def add_deviation_safe(df: pl.DataFrame, baseline: pl.DataFrame, base_feature_cols: list[str]) -> pl.DataFrame:
    out = df.join(baseline, on="user", how="left")

    eps = 1e-3  # much bigger than 1e-6 to avoid explosion

    for c in base_feature_cols:
        base_c = f"base_{c}"

        # diff is always safe and useful
        out = out.with_columns(
            (pl.col(c) - pl.col(base_c)).fill_null(0).alias(f"diff_{c}")
        )

        # decide if ratio should be created
        skip_ratio = (c in NO_RATIO_EXACT) or c.startswith(NO_RATIO_PREFIXES)

        if not skip_ratio:
            # log-ratio with denominator clamped away from 0
            # log1p(current) - log1p(max(base, eps))
            out = out.with_columns(
                (pl.max_horizontal(pl.col(base_c), pl.lit(eps)).log1p()
                 ).alias(f"_tmp_base_log_{c}")
            ).with_columns(
                (pl.max_horizontal(pl.col(c), pl.lit(0)).log1p()
                 ).alias(f"_tmp_cur_log_{c}")
            ).with_columns(
                (pl.col(f"_tmp_cur_log_{c}") - pl.col(f"_tmp_base_log_{c}"))
                .clip(-10, 10)  # hard clip to stop extreme tails
                .fill_null(0)
                .alias(f"log_ratio_{c}")
            ).drop([f"_tmp_base_log_{c}", f"_tmp_cur_log_{c}"])

    # drop base columns
    out = out.drop([f"base_{c}" for c in base_feature_cols])
    return out

In [6]:
# base_feature_cols should be the features from train_df (before deviation)
base_feature_cols = [c for c in train_df.columns if c != "user"]

baseline = (
    train_df.select(["user"] + base_feature_cols)
            .group_by("user")
            .agg([pl.col(c).mean().alias(f"base_{c}") for c in base_feature_cols])
)

train_df2 = add_deviation_safe(train_df, baseline, base_feature_cols)
val_df2   = add_deviation_safe(val_df, baseline, base_feature_cols)
test_df2  = add_deviation_safe(test_df, baseline, base_feature_cols)

In [7]:
non_feature_cols = {"user", "label", "compromise_events", "src_user"}

feature_cols2 = [
    c for c, dtype in zip(train_df2.columns, train_df2.dtypes)
    if c not in non_feature_cols and str(dtype) in ("Float64","Float32","Int64","Int32","Int16","Int8")
]

print("num features:", len(feature_cols2))

num features: 55


In [8]:
import numpy as np
from sklearn.preprocessing import RobustScaler

X_train = train_df2.select(feature_cols2).to_numpy()
scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_train_s = np.nan_to_num(X_train_s, nan=0.0, posinf=0.0, neginf=0.0)

print("Min/Max scaled train:", X_train_s.min(), X_train_s.max())

Min/Max scaled train: -2.6974908423170487 1645026.0


In [10]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import RobustScaler
from torch.utils.data import Dataset, DataLoader

X_train = train_df2.select(feature_cols2).to_numpy()
X_val   = val_df2.select(feature_cols2).to_numpy()
X_test  = test_df2.select(feature_cols2).to_numpy()

scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

input_dim = X_train_s.shape[1]
print("input_dim:", input_dim)

class NumpyDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

train_loader = DataLoader(NumpyDataset(X_train_s), batch_size=4096, shuffle=True, num_workers=0)
val_tensor   = torch.tensor(X_val_s, dtype=torch.float32)
test_tensor  = torch.tensor(X_test_s, dtype=torch.float32)

class AutoEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, latent_dim=16, dropout=0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

def train_ae(train_loader, val_tensor, input_dim, hidden_dim, latent_dim, dropout, lr, weight_decay, epochs=40, patience=5):
    model = AutoEncoder(input_dim, hidden_dim, latent_dim, dropout).to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    bad = 0

    for _ in range(epochs):
        model.train()
        for xb in train_loader:
            xb = xb.to(device)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), xb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            xv = val_tensor.to(device)
            val_loss = loss_fn(model(xv), xv).item()

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val

def recon_scores_mae(model, X_tensor):
    model.eval()
    with torch.no_grad():
        x = X_tensor.to(device)
        r = model(x)
        return (r - x).abs().mean(dim=1).detach().cpu().numpy()

# sweep (unsupervised selection)
search_space = {
    "hidden_dim": [64, 128, 256],
    "latent_dim": [8, 16, 32],
    "dropout":    [0.0, 0.1, 0.2],
    "lr":         [1e-3, 3e-4],
    "weight_decay":[0.0, 1e-4],
}

best_model = None
best_cfg = None
best_val_loss = float("inf")

total = (
    len(search_space["hidden_dim"]) *
    len(search_space["latent_dim"]) *
    len(search_space["dropout"]) *
    len(search_space["lr"]) *
    len(search_space["weight_decay"])
)
trial = 0

for hd in search_space["hidden_dim"]:
    for ld in search_space["latent_dim"]:
        for dr in search_space["dropout"]:
            for lr in search_space["lr"]:
                for wd in search_space["weight_decay"]:
                    trial += 1
                    model, vloss = train_ae(train_loader, val_tensor, input_dim, hd, ld, dr, lr, wd)
                    if vloss < best_val_loss:
                        best_val_loss = vloss
                        best_model = model
                        best_cfg = {"hidden_dim": hd, "latent_dim": ld, "dropout": dr, "lr": lr, "weight_decay": wd}
                        print(f"[{trial}/{total}] NEW BEST val_mse={best_val_loss:.6f} cfg={best_cfg}")
                    elif trial % 10 == 0:
                        print(f"[{trial}/{total}] current best val_mse={best_val_loss:.6f}")

print("\nFINAL BEST AE CONFIG:", best_cfg)
print("FINAL BEST VAL MSE:", best_val_loss)

# show top anomalies in TEST by reconstruction error
test_scores = recon_scores_mae(best_model, test_tensor)
top_idx = np.argsort(test_scores)[::-1][:20]
print("\nTop 20 test anomalies (by AE recon MAE):")
print(top_idx)
print(test_scores[top_idx])

input_dim: 55
device: cpu
[1/108] NEW BEST val_mse=363994624.000000 cfg={'hidden_dim': 64, 'latent_dim': 8, 'dropout': 0.0, 'lr': 0.001, 'weight_decay': 0.0}
[2/108] NEW BEST val_mse=363188832.000000 cfg={'hidden_dim': 64, 'latent_dim': 8, 'dropout': 0.0, 'lr': 0.001, 'weight_decay': 0.0001}
[3/108] NEW BEST val_mse=361253504.000000 cfg={'hidden_dim': 64, 'latent_dim': 8, 'dropout': 0.0, 'lr': 0.0003, 'weight_decay': 0.0}
[4/108] NEW BEST val_mse=359347648.000000 cfg={'hidden_dim': 64, 'latent_dim': 8, 'dropout': 0.0, 'lr': 0.0003, 'weight_decay': 0.0001}
[8/108] NEW BEST val_mse=358189440.000000 cfg={'hidden_dim': 64, 'latent_dim': 8, 'dropout': 0.1, 'lr': 0.0003, 'weight_decay': 0.0001}
[10/108] current best val_mse=358189440.000000
[19/108] NEW BEST val_mse=354996672.000000 cfg={'hidden_dim': 64, 'latent_dim': 16, 'dropout': 0.1, 'lr': 0.0003, 'weight_decay': 0.0}
[20/108] current best val_mse=354996672.000000
[30/108] current best val_mse=354996672.000000
[40/108] current best val_

In [ ]:
print("Any NaN in train:", np.isnan(X_train_s).any(), "Any inf:", np.isinf(X_train_s).any())
print("Any NaN in val:", np.isnan(X_val_s).any(), "Any inf:", np.isinf(X_val_s).any())
print("Min/Max train:", X_train_s.min(), X_train_s.max())

In [11]:
import numpy as np

# X_train_s is your scaled numpy array
min_pos = np.unravel_index(np.argmin(X_train_s), X_train_s.shape)
row_i, col_j = min_pos
print("Min value:", X_train_s[row_i, col_j])
print("Row index:", row_i, "Col index:", col_j)
print("Feature name:", feature_cols2[col_j])

# show that row's top 10 most extreme values (by abs)
row_vals = X_train_s[row_i]
top_idx = np.argsort(np.abs(row_vals))[::-1][:10]
print([(feature_cols2[i], float(row_vals[i])) for i in top_idx])

Min value: -2.6974908423170487
Row index: 81 Col index: 14
Feature name: top_logon_share
[('median_dt', 2053.0), ('p90_dt', 675.0268096514745), ('mean_dt', 193.70524739296374), ('src_per_login', 27.605585520431426), ('dst_per_login', 26.07292083963318), ('std_dt', 21.74750898359313), ('top_logon_share', -2.6974908423170487), ('top_dst_share', 1.140907230558712), ('dst_mean_cnt', -0.6251627999013033), ('night_rate', -0.6089393565620271)]


In [ ]:
CLIP = 15.0  # start with 10-20
X_train_s = np.clip(X_train_s, -CLIP, CLIP)
X_val_s   = np.clip(X_val_s,   -CLIP, CLIP)
X_test_s  = np.clip(X_test_s,  -CLIP, CLIP)

print("Min/Max after clip:", X_train_s.min(), X_train_s.max())

SANITY CHECK

In [12]:
import polars as pl
import numpy as np

# add scores back to test_df2 (keep same row order)
test_scored = test_df2.with_columns(
    pl.Series("ae_score", test_scores)
)

# choose a few interpretable columns to inspect
cols_to_show = [
    "user",
    "ae_score",
    "total_logins",
    "failed_logins",
    "fail_rate",
    "unique_destinations",
    "unique_src_computers",
    "median_dt",
    "std_dt",
    "top_dst_share",
    "unique_logon_types",
]

cols_to_show = [c for c in cols_to_show if c in test_scored.columns]

test_scored.sort("ae_score", descending=True).select(cols_to_show).head(20)

user,ae_score,total_logins,failed_logins,fail_rate,unique_destinations,unique_src_computers,median_dt,std_dt,top_dst_share,unique_logon_types
str,f32,u32,i64,f64,u32,u32,f64,f64,f64,u32
"""U22""",419178.375,1094327,0,0.0,23,18,0.0,0.795734,0.198633,2
"""U66""",232588.921875,696649,0,0.0,144,104,0.0,0.919806,0.062168,2
"""U346""",119067.179688,50774,0,0.0,10,7,0.0,183.414011,0.774274,3
"""U7832""",101688.570312,9,0,0.0,2,2,7.0,31326.922297,0.888889,2
"""U292""",91630.492188,263883,0,0.0,39,25,0.0,3.37262,0.052614,3
…,…,…,…,…,…,…,…,…,…,…
"""U6""",51593.417969,144407,0,0.0,22,17,1.0,3.33559,0.382634,4
"""U54""",46812.386719,102173,0,0.0,8,6,2.0,3.854766,0.998199,2
"""U75""",38954.113281,1144,0,0.0,5,5,16.0,428.68314,0.645979,3


In [13]:
drop_if_contains = [
    "total_logins", "failed_logins",
    "dst_total_cnt", "top_dst_cnt",
    "logon_total_cnt", "top_logon_cnt",
]

feature_cols2_small = [
    c for c in feature_cols2
    if not any(key == c or c.endswith(key) or key in c for key in drop_if_contains)
]

print("features before:", len(feature_cols2), "after:", len(feature_cols2_small))

features before: 55 after: 42


In [14]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
import numpy as np

X_train = train_df2.select(feature_cols2).to_numpy()
X_test  = test_df2.select(feature_cols2).to_numpy()

scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

X_train_s = np.nan_to_num(X_train_s, nan=0.0, posinf=0.0, neginf=0.0)
X_test_s  = np.nan_to_num(X_test_s,  nan=0.0, posinf=0.0, neginf=0.0)

X_train_s = np.clip(X_train_s, -15, 15)
X_test_s  = np.clip(X_test_s,  -15, 15)

iso = IsolationForest(n_estimators=500, contamination=0.01, random_state=42, n_jobs=-1)
iso.fit(X_train_s)
if_scores = -iso.decision_function(X_test_s)

# correlation between rankings
corr = np.corrcoef(if_scores, test_scores)[0, 1]
print("Correlation(IF score, AE score):", corr)

# overlap of top-K flagged users
def topk_idx(scores, k):
    return set(np.argsort(scores)[::-1][:k])

for k in [50, 100, 200]:
    ov = len(topk_idx(if_scores, k) & topk_idx(test_scores, k))
    print(f"Top-{k} overlap:", ov, "/", k)

Correlation(IF score, AE score): 0.21399725301004316
Top-50 overlap: 21 / 50
Top-100 overlap: 35 / 100
Top-200 overlap: 64 / 200


In [15]:
# ============================================================
# FULL PIPELINE (ONE BLOCK)
# Updated features + safe deviation/log-ratio + AE (structure-only)
# Notes:
# - Builds user-level features from filtered_auth.parquet
# - Splits by time quantiles (70/15/15)
# - Adds TRAIN-only diff_* and log_ratio_* features safely
# - Trains an AutoEncoder on structure-only features (drops raw volume counts)
# - Prints top anomalies with interpretable columns
# ============================================================

import polars as pl
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import RobustScaler
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# 0) Helpers
# ----------------------------
def norm_user(expr: pl.Expr) -> pl.Expr:
    return (
        expr.cast(pl.Utf8)
            .str.strip_chars()
            .str.split("@").list.first()
            .str.split("\\").list.last()
    )

def user_features(ldf: pl.LazyFrame) -> pl.LazyFrame:
    ldf = (
        ldf.filter(pl.col("src_user").str.starts_with("U"))
           .with_columns([
               norm_user(pl.col("src_user")).alias("user"),
               pl.col("time").cast(pl.Int64),
               ((pl.col("time") // 3600) % 24).cast(pl.Int32).alias("hour"),
               ((pl.col("time") // 86400) % 7).cast(pl.Int32).alias("dow"),
           ])
           .with_columns([
               (pl.col("dow") >= 5).cast(pl.Int8).alias("is_weekend"),
               (pl.col("hour").is_between(0, 5)).cast(pl.Int8).alias("is_night"),
               (pl.col("status") == "Failure").cast(pl.Int8).alias("is_fail"),
           ])
    )

    core = (
        ldf.group_by("user")
           .agg([
               pl.len().alias("total_logins"),
               pl.col("is_fail").sum().alias("failed_logins"),
               pl.col("dst_computer").n_unique().alias("unique_destinations"),
               pl.col("src_computer").n_unique().alias("unique_src_computers"),
               pl.col("is_fail").mean().alias("fail_rate"),
               pl.col("is_night").mean().alias("night_rate"),
               pl.col("is_weekend").mean().alias("weekend_rate"),
               pl.col("hour").n_unique().alias("unique_hours"),
           ])
           .with_columns([
               (pl.col("failed_logins") / (pl.col("total_logins") + 1e-6)).alias("fail_rate_2"),
               (pl.col("unique_destinations") / (pl.col("total_logins") + 1e-6)).alias("dst_per_login"),
               (pl.col("unique_src_computers") / (pl.col("total_logins") + 1e-6)).alias("src_per_login"),
           ])
    )

    gaps = (
        ldf.select(["user", "time"])
           .sort(["user", "time"])
           .with_columns([(pl.col("time") - pl.col("time").shift(1)).over("user").alias("dt")])
           .group_by("user")
           .agg([
               pl.col("dt").drop_nulls().median().alias("median_dt"),
               pl.col("dt").drop_nulls().mean().alias("mean_dt"),
               pl.col("dt").drop_nulls().std().alias("std_dt"),
               pl.col("dt").drop_nulls().quantile(0.90).alias("p90_dt"),
           ])
    )

    dest_share = (
        ldf.group_by(["user", "dst_computer"])
           .agg(pl.len().alias("cnt"))
           .group_by("user")
           .agg([
               pl.col("cnt").max().alias("top_dst_cnt"),
               pl.col("cnt").sum().alias("dst_total_cnt"),
               pl.col("cnt").mean().alias("dst_mean_cnt"),
               pl.col("cnt").std().alias("dst_std_cnt"),
           ])
           .with_columns([(pl.col("top_dst_cnt") / (pl.col("dst_total_cnt") + 1e-6)).alias("top_dst_share")])
    )

    logon_div = (
        ldf.group_by(["user", "logon_type"])
           .agg(pl.len().alias("lt_cnt"))
           .group_by("user")
           .agg([
               pl.col("logon_type").n_unique().alias("unique_logon_types"),
               pl.col("lt_cnt").max().alias("top_logon_cnt"),
               pl.col("lt_cnt").sum().alias("logon_total_cnt"),
           ])
           .with_columns([(pl.col("top_logon_cnt") / (pl.col("logon_total_cnt") + 1e-6)).alias("top_logon_share")])
    )

    feat = (
        core.join(gaps, on="user", how="left")
            .join(dest_share, on="user", how="left")
            .join(logon_div, on="user", how="left")
            .with_columns(pl.all().exclude("user").fill_null(0))
    )
    return feat

NO_RATIO_PREFIXES = ("median_dt", "mean_dt", "std_dt", "p90_dt")
NO_RATIO_EXACT = {"fail_rate", "fail_rate_2", "night_rate", "weekend_rate"}

def add_deviation_safe(df: pl.DataFrame, baseline: pl.DataFrame, base_feature_cols: list[str]) -> pl.DataFrame:
    out = df.join(baseline, on="user", how="left")

    eps = 1e-3

    for c in base_feature_cols:
        base_c = f"base_{c}"

        out = out.with_columns(
            (pl.col(c) - pl.col(base_c)).fill_null(0).alias(f"diff_{c}")
        )

        skip_ratio = (c in NO_RATIO_EXACT) or c.startswith(NO_RATIO_PREFIXES)
        if not skip_ratio:
            out = out.with_columns(
                (pl.max_horizontal(pl.col(base_c), pl.lit(eps)).log1p()).alias(f"_tmp_base_log_{c}")
            ).with_columns(
                (pl.max_horizontal(pl.col(c), pl.lit(0)).log1p()).alias(f"_tmp_cur_log_{c}")
            ).with_columns(
                (pl.col(f"_tmp_cur_log_{c}") - pl.col(f"_tmp_base_log_{c}"))
                .clip(-10, 10)
                .fill_null(0)
                .alias(f"log_ratio_{c}")
            ).drop([f"_tmp_base_log_{c}", f"_tmp_cur_log_{c}"])

    out = out.drop([f"base_{c}" for c in base_feature_cols])
    return out

# ----------------------------
# 1) Load + split by time
# ----------------------------
auth = pl.scan_parquet("filtered_auth.parquet")

times = auth.select(pl.col("time").cast(pl.Int64)).collect()
q70 = int(times["time"].quantile(0.70))
q85 = int(times["time"].quantile(0.85))
print("q70:", q70, "q85:", q85)

train_auth = auth.filter(pl.col("time") <= q70)
val_auth   = auth.filter((pl.col("time") > q70) & (pl.col("time") <= q85))
test_auth  = auth.filter(pl.col("time") > q85)

# ----------------------------
# 2) Build features per split
# ----------------------------
train_df = user_features(train_auth).collect()
val_df   = user_features(val_auth).collect()
test_df  = user_features(test_auth).collect()

print("Users train/val/test:", train_df.height, val_df.height, test_df.height)

# ----------------------------
# 3) TRAIN-only baselines + safe diff/log-ratio
# ----------------------------
base_feature_cols = [c for c in train_df.columns if c != "user"]

baseline = (
    train_df.select(["user"] + base_feature_cols)
            .group_by("user")
            .agg([pl.col(c).mean().alias(f"base_{c}") for c in base_feature_cols])
)

train_df2 = add_deviation_safe(train_df, baseline, base_feature_cols)
val_df2   = add_deviation_safe(val_df, baseline, base_feature_cols)
test_df2  = add_deviation_safe(test_df, baseline, base_feature_cols)

# numeric feature columns
non_feature_cols = {"user", "label", "compromise_events", "src_user"}
feature_cols2 = [
    c for c, dtype in zip(train_df2.columns, train_df2.dtypes)
    if c not in non_feature_cols and str(dtype) in ("Float64","Float32","Int64","Int32","Int16","Int8")
]
print("All numeric features:", len(feature_cols2))

# ----------------------------
# 4) AE feature selection: structure-only (drop volume-ish columns)
# ----------------------------
drop_keywords = [
    "total_logins",
    "failed_logins",
    "_total_cnt",
    "_cnt",
    "top_dst_cnt",
    "top_logon_cnt",
]

feature_cols_ae = [
    c for c in feature_cols2
    if not any(key in c for key in drop_keywords)
]
print("AE structure features:", len(feature_cols_ae))

# ----------------------------
# 5) Build matrices + scale + sanitize + clip
# ----------------------------
X_train = train_df2.select(feature_cols_ae).to_numpy()
X_val   = val_df2.select(feature_cols_ae).to_numpy()
X_test  = test_df2.select(feature_cols_ae).to_numpy()

print("Rows train/val/test (AE):", X_train.shape[0], X_val.shape[0], X_test.shape[0])

# if VAL empty, use slice of train
if X_val.shape[0] == 0:
    X_val = X_train[: min(5000, X_train.shape[0])].copy()
    print("VAL was empty. Using pseudo-VAL rows:", X_val.shape[0])

scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

X_train_s = np.nan_to_num(X_train_s, nan=0.0, posinf=0.0, neginf=0.0)
X_val_s   = np.nan_to_num(X_val_s,   nan=0.0, posinf=0.0, neginf=0.0)
X_test_s  = np.nan_to_num(X_test_s,  nan=0.0, posinf=0.0, neginf=0.0)

CLIP = 15.0
X_train_s = np.clip(X_train_s, -CLIP, CLIP)
X_val_s   = np.clip(X_val_s,   -CLIP, CLIP)
X_test_s  = np.clip(X_test_s,  -CLIP, CLIP)

input_dim = X_train_s.shape[1]
print("input_dim:", input_dim)

# ----------------------------
# 6) AutoEncoder training + selection by lowest VAL MSE
# ----------------------------
class NumpyDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

train_loader = DataLoader(NumpyDataset(X_train_s), batch_size=4096, shuffle=True, num_workers=0)
val_tensor   = torch.tensor(X_val_s, dtype=torch.float32)
test_tensor  = torch.tensor(X_test_s, dtype=torch.float32)

class AutoEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=16, dropout=0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

def train_ae(train_loader, val_tensor, input_dim, hidden_dim, latent_dim, dropout, lr, weight_decay, epochs=40, patience=5):
    model = AutoEncoder(input_dim, hidden_dim, latent_dim, dropout).to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    bad = 0

    for _ in range(epochs):
        model.train()
        for xb in train_loader:
            xb = xb.to(device)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), xb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            xv = val_tensor.to(device)
            val_loss = loss_fn(model(xv), xv).item() if xv.shape[0] > 0 else float("inf")

        if not np.isfinite(val_loss):
            val_loss = float("inf")

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val

def recon_scores_mae(model, X_tensor):
    model.eval()
    with torch.no_grad():
        x = X_tensor.to(device)
        r = model(x)
        return (r - x).abs().mean(dim=1).detach().cpu().numpy()

search_space = {
    "hidden_dim": [64, 128],
    "latent_dim": [8, 16],
    "dropout":    [0.0, 0.1, 0.2],
    "lr":         [3e-4, 1e-3],
    "weight_decay":[0.0, 1e-4],
}

best_model = None
best_cfg = None
best_val_loss = float("inf")

total = (
    len(search_space["hidden_dim"]) *
    len(search_space["latent_dim"]) *
    len(search_space["dropout"]) *
    len(search_space["lr"]) *
    len(search_space["weight_decay"])
)
trial = 0

for hd in search_space["hidden_dim"]:
    for ld in search_space["latent_dim"]:
        for dr in search_space["dropout"]:
            for lr in search_space["lr"]:
                for wd in search_space["weight_decay"]:
                    trial += 1
                    model, vloss = train_ae(train_loader, val_tensor, input_dim, hd, ld, dr, lr, wd)
                    if best_model is None or vloss < best_val_loss:
                        best_val_loss = vloss
                        best_model = model
                        best_cfg = {"hidden_dim": hd, "latent_dim": ld, "dropout": dr, "lr": lr, "weight_decay": wd}
                        print(f"[{trial}/{total}] NEW BEST val_mse={best_val_loss:.6f} cfg={best_cfg}")
                    elif trial % 10 == 0:
                        print(f"[{trial}/{total}] current best val_mse={best_val_loss:.6f}")

print("\nFINAL BEST AE CONFIG:", best_cfg)
print("FINAL BEST VAL MSE:", best_val_loss)

# ----------------------------
# 7) Score test + show top anomalies (interpretable)
# ----------------------------
test_scores = recon_scores_mae(best_model, test_tensor)

test_scored = test_df2.with_columns(pl.Series("ae_score", test_scores))
cols_to_show = [
    "user", "ae_score",
    "total_logins", "failed_logins", "fail_rate",
    "unique_destinations", "unique_src_computers",
    "median_dt", "std_dt", "top_dst_share",
    "unique_logon_types", "top_logon_share",
]
cols_to_show = [c for c in cols_to_show if c in test_scored.columns]

print("\nTop 20 test anomalies (AE):")
print(test_scored.sort("ae_score", descending=True).select(cols_to_show).head(20))

q70: 1853062 q85: 2234039
Users train/val/test: 11376 10018 10225
All numeric features: 55
AE structure features: 36
Rows train/val/test (AE): 11376 10018 10225
input_dim: 36
device: cpu
[1/48] NEW BEST val_mse=21.917097 cfg={'hidden_dim': 64, 'latent_dim': 8, 'dropout': 0.0, 'lr': 0.0003, 'weight_decay': 0.0}
[5/48] NEW BEST val_mse=21.644236 cfg={'hidden_dim': 64, 'latent_dim': 8, 'dropout': 0.1, 'lr': 0.0003, 'weight_decay': 0.0}
[10/48] current best val_mse=21.644236
[17/48] NEW BEST val_mse=21.591915 cfg={'hidden_dim': 64, 'latent_dim': 16, 'dropout': 0.1, 'lr': 0.0003, 'weight_decay': 0.0}
[20/48] current best val_mse=21.591915
[30/48] current best val_mse=21.591915
[40/48] current best val_mse=21.591915

FINAL BEST AE CONFIG: {'hidden_dim': 64, 'latent_dim': 16, 'dropout': 0.1, 'lr': 0.0003, 'weight_decay': 0.0}
FINAL BEST VAL MSE: 21.591915130615234

Top 20 test anomalies (AE):
shape: (20, 12)
┌────────┬──────────┬────────────┬────────────┬───┬────────────┬───────────┬─────────

IMPROVE THIS

In [17]:
dt_cols = {"median_dt", "mean_dt", "std_dt", "p90_dt"}
feature_cols_ae2 = [c for c in feature_cols_ae if c not in dt_cols and not c.endswith(tuple(dt_cols))]

print("AE features before:", len(feature_cols_ae))
print("AE features after dropping dt:", len(feature_cols_ae2))
import numpy as np
import polars as pl

MIN_LOGINS = 50  # try 50, 100

test_scored = test_df2.with_columns(pl.Series("ae_score", test_scores))

test_scored_active = test_scored.filter(pl.col("total_logins") >= MIN_LOGINS)

print("Users in test:", test_scored.height)
print("Users scored (>=MIN_LOGINS):", test_scored_active.height)

cols_to_show = [
    "user", "ae_score",
    "total_logins", "failed_logins", "fail_rate",
    "unique_destinations", "unique_src_computers",
    "top_dst_share", "unique_logon_types", "top_logon_share",
]
cols_to_show = [c for c in cols_to_show if c in test_scored_active.columns]

print(test_scored_active.sort("ae_score", descending=True).select(cols_to_show).head(20))


AE features before: 36
AE features after dropping dt: 28
Users in test: 10225
Users scored (>=MIN_LOGINS): 8980
shape: (20, 10)
┌────────┬──────────┬────────────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ user   ┆ ae_score ┆ total_logi ┆ failed_log ┆ … ┆ unique_src ┆ top_dst_s ┆ unique_lo ┆ top_logon │
│ ---    ┆ ---      ┆ ns         ┆ ins        ┆   ┆ _computers ┆ hare      ┆ gon_types ┆ _share    │
│ str    ┆ f32      ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---       ┆ ---       ┆ ---       │
│        ┆          ┆ u32        ┆ i64        ┆   ┆ u32        ┆ f64       ┆ u32       ┆ f64       │
╞════════╪══════════╪════════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ U456   ┆ 4.255187 ┆ 78         ┆ 0          ┆ … ┆ 59         ┆ 0.102564  ┆ 4         ┆ 0.807692  │
│ U11223 ┆ 3.901954 ┆ 9308       ┆ 0          ┆ … ┆ 22         ┆ 0.819403  ┆ 4         ┆ 0.923829  │
│ U6836  ┆ 3.785136 ┆ 144        ┆ 0          ┆ … ┆ 3          ┆

In [18]:
# ============================================================
# IF + AE COMBO SCORING (with correlation + top alerts)
# Assumes you already have:
# - train_df2, val_df2, test_df2 (Polars DataFrames)
# - feature_cols2 (all numeric features for IF)
# - feature_cols_ae (structure-only AE features) OR feature_cols_ae2 (dt-removed)
# - best_model (trained AE) AND scaler used for AE, OR you can retrain AE first
#
# This block:
# 1) Fits Isolation Forest on TRAIN (full feature set)
# 2) Scores TEST with IF
# 3) Scores TEST with AE
# 4) Normalizes both scores using VAL stats (z-score)
# 5) Combines scores
# 6) Filters to active users (optional)
# 7) Prints correlation + overlap + top alerts
# ============================================================

import numpy as np
import polars as pl
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest

# ----------------------------
# 0) Utilities
# ----------------------------
def zscore_with_ref(x, ref):
    mu = ref.mean()
    sd = ref.std() + 1e-8
    return (x - mu) / sd

def topk_idx(scores, k):
    return set(np.argsort(scores)[::-1][:k])

def recon_scores_mae(model, X_tensor, device="cpu"):
    model.eval()
    import torch
    with torch.no_grad():
        x = X_tensor.to(device)
        r = model(x)
        return (r - x).abs().mean(dim=1).detach().cpu().numpy()

# ----------------------------
# 1) Isolation Forest scores (full features)
# ----------------------------
X_train_if = train_df2.select(feature_cols2).to_numpy()
X_val_if   = val_df2.select(feature_cols2).to_numpy()
X_test_if  = test_df2.select(feature_cols2).to_numpy()

scaler_if = RobustScaler()
X_train_if_s = scaler_if.fit_transform(X_train_if)
X_val_if_s   = scaler_if.transform(X_val_if)
X_test_if_s  = scaler_if.transform(X_test_if)

X_train_if_s = np.nan_to_num(X_train_if_s, nan=0.0, posinf=0.0, neginf=0.0)
X_val_if_s   = np.nan_to_num(X_val_if_s,   nan=0.0, posinf=0.0, neginf=0.0)
X_test_if_s  = np.nan_to_num(X_test_if_s,  nan=0.0, posinf=0.0, neginf=0.0)

# optional clip for stability
CLIP = 15.0
X_train_if_s = np.clip(X_train_if_s, -CLIP, CLIP)
X_val_if_s   = np.clip(X_val_if_s,   -CLIP, CLIP)
X_test_if_s  = np.clip(X_test_if_s,  -CLIP, CLIP)

iso = IsolationForest(n_estimators=500, contamination=0.01, random_state=42, n_jobs=-1)
iso.fit(X_train_if_s)

val_if_score  = -iso.decision_function(X_val_if_s)
test_if_score = -iso.decision_function(X_test_if_s)

# ----------------------------
# 2) Autoencoder scores (structure-only features)
# Use dt-removed if you made it:
# feature_cols_ae_used = feature_cols_ae2
# else:
# feature_cols_ae_used = feature_cols_ae
# ----------------------------
feature_cols_ae_used = feature_cols_ae  # change to feature_cols_ae2 if you created it

X_train_ae = train_df2.select(feature_cols_ae_used).to_numpy()
X_val_ae   = val_df2.select(feature_cols_ae_used).to_numpy()
X_test_ae  = test_df2.select(feature_cols_ae_used).to_numpy()

scaler_ae = RobustScaler()
X_train_ae_s = scaler_ae.fit_transform(X_train_ae)
X_val_ae_s   = scaler_ae.transform(X_val_ae)
X_test_ae_s  = scaler_ae.transform(X_test_ae)

X_train_ae_s = np.nan_to_num(X_train_ae_s, nan=0.0, posinf=0.0, neginf=0.0)
X_val_ae_s   = np.nan_to_num(X_val_ae_s,   nan=0.0, posinf=0.0, neginf=0.0)
X_test_ae_s  = np.nan_to_num(X_test_ae_s,  nan=0.0, posinf=0.0, neginf=0.0)

X_train_ae_s = np.clip(X_train_ae_s, -CLIP, CLIP)
X_val_ae_s   = np.clip(X_val_ae_s,   -CLIP, CLIP)
X_test_ae_s  = np.clip(X_test_ae_s,  -CLIP, CLIP)

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

val_tensor_ae  = torch.tensor(X_val_ae_s, dtype=torch.float32)
test_tensor_ae = torch.tensor(X_test_ae_s, dtype=torch.float32)

# best_model should already exist from your AE sweep
val_ae_score  = recon_scores_mae(best_model, val_tensor_ae, device=device)
test_ae_score = recon_scores_mae(best_model, test_tensor_ae, device=device)

# ----------------------------
# 3) Normalize using VAL stats and combine
# ----------------------------
val_if_z  = zscore_with_ref(val_if_score, val_if_score)
val_ae_z  = zscore_with_ref(val_ae_score, val_ae_score)
test_if_z = zscore_with_ref(test_if_score, val_if_score)
test_ae_z = zscore_with_ref(test_ae_score, val_ae_score)

w_if = 0.7
w_ae = 0.3
test_combo = w_if * test_if_z + w_ae * test_ae_z

# ----------------------------
# 4) Diagnostics: correlation + top-k overlap
# ----------------------------
corr = np.corrcoef(test_if_score, test_ae_score)[0, 1]
print("Correlation(IF, AE) on TEST:", corr)

for k in [50, 100, 200]:
    ov = len(topk_idx(test_if_score, k) & topk_idx(test_ae_score, k))
    print(f"Top-{k} overlap:", ov, "/", k)

# ----------------------------
# 5) Put scores back into a table + optional active filter
# ----------------------------
test_scored = test_df2.with_columns([
    pl.Series("if_score", test_if_score),
    pl.Series("ae_score", test_ae_score),
    pl.Series("combo_score", test_combo),
])

MIN_LOGINS = 50  # tune this
if "total_logins" in test_scored.columns:
    test_scored = test_scored.filter(pl.col("total_logins") >= MIN_LOGINS)

cols_to_show = [
    "user", "combo_score", "if_score", "ae_score",
    "total_logins", "failed_logins", "fail_rate",
    "unique_destinations", "unique_src_computers",
    "dst_per_login", "src_per_login",
    "top_dst_share", "unique_logon_types", "top_logon_share",
    "median_dt", "std_dt"
]
cols_to_show = [c for c in cols_to_show if c in test_scored.columns]

print("\nTop 30 alerts by COMBO score:")
print(test_scored.sort("combo_score", descending=True).select(cols_to_show).head(30))

Correlation(IF, AE) on TEST: 0.5074628469834946
Top-50 overlap: 2 / 50
Top-100 overlap: 5 / 100
Top-200 overlap: 14 / 200

Top 30 alerts by COMBO score:
shape: (30, 16)
┌────────┬────────────┬───────────┬──────────┬───┬────────────┬────────────┬───────────┬───────────┐
│ user   ┆ combo_scor ┆ if_score  ┆ ae_score ┆ … ┆ unique_log ┆ top_logon_ ┆ median_dt ┆ std_dt    │
│ ---    ┆ e          ┆ ---       ┆ ---      ┆   ┆ on_types   ┆ share      ┆ ---       ┆ ---       │
│ str    ┆ ---        ┆ f64       ┆ f32      ┆   ┆ ---        ┆ ---        ┆ f64       ┆ f64       │
│        ┆ f64        ┆           ┆          ┆   ┆ u32        ┆ f64        ┆           ┆           │
╞════════╪════════════╪═══════════╪══════════╪═══╪════════════╪════════════╪═══════════╪═══════════╡
│ U4965  ┆ 3.03666    ┆ 0.054584  ┆ 3.108054 ┆ … ┆ 1          ┆ 1.0        ┆ 16.0      ┆ 14.810024 │
│ U456   ┆ 2.804209   ┆ -0.01315  ┆ 4.255187 ┆ … ┆ 4          ┆ 0.807692   ┆ 902.0     ┆ 9625.4392 │
│        ┆            ┆

In [ ]:
dt_keys = ["median_dt", "mean_dt", "std_dt", "p90_dt"]

feature_cols_ae2 = [
    c for c in feature_cols_ae
    if not any(k == c or c.endswith(k) or k in c for k in dt_keys)
]

print("AE features before:", len(feature_cols_ae))
print("AE features after dt-drop:", len(feature_cols_ae2))


In [19]:
# ============================================================
# FULL CODE: dt-stable features + safe deviation/log-ratio
# + train AE WITHOUT dt features
# + compute IF + AE combo score
# + print correlation/overlap + top alerts
#
# Requirements:
# - filtered_auth.parquet exists with columns:
#   time, src_user, src_computer, dst_computer, logon_type, status
# ============================================================

import polars as pl
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# 0) Helpers
# ----------------------------
def norm_user(expr: pl.Expr) -> pl.Expr:
    return (
        expr.cast(pl.Utf8)
            .str.strip_chars()
            .str.split("@").list.first()
            .str.split("\\").list.last()
    )

def user_features(ldf: pl.LazyFrame, dt_min_events: int = 30) -> pl.LazyFrame:
    ldf = (
        ldf.filter(pl.col("src_user").str.starts_with("U"))
           .with_columns([
               norm_user(pl.col("src_user")).alias("user"),
               pl.col("time").cast(pl.Int64),
               ((pl.col("time") // 3600) % 24).cast(pl.Int32).alias("hour"),
               ((pl.col("time") // 86400) % 7).cast(pl.Int32).alias("dow"),
           ])
           .with_columns([
               (pl.col("dow") >= 5).cast(pl.Int8).alias("is_weekend"),
               (pl.col("hour").is_between(0, 5)).cast(pl.Int8).alias("is_night"),
               (pl.col("status") == "Failure").cast(pl.Int8).alias("is_fail"),
           ])
    )

    core = (
        ldf.group_by("user")
           .agg([
               pl.len().alias("total_logins"),
               pl.col("is_fail").sum().alias("failed_logins"),
               pl.col("dst_computer").n_unique().alias("unique_destinations"),
               pl.col("src_computer").n_unique().alias("unique_src_computers"),
               pl.col("is_fail").mean().alias("fail_rate"),
               pl.col("is_night").mean().alias("night_rate"),
               pl.col("is_weekend").mean().alias("weekend_rate"),
               pl.col("hour").n_unique().alias("unique_hours"),
           ])
           .with_columns([
               (pl.col("failed_logins") / (pl.col("total_logins") + 1e-6)).alias("fail_rate_2"),
               (pl.col("unique_destinations") / (pl.col("total_logins") + 1e-6)).alias("dst_per_login"),
               (pl.col("unique_src_computers") / (pl.col("total_logins") + 1e-6)).alias("src_per_login"),
           ])
    )

    # dt stats with minimum-events gating (prevents sparse-user explosions)
    gaps = (
        ldf.select(["user", "time"])
           .sort(["user", "time"])
           .with_columns([(pl.col("time") - pl.col("time").shift(1)).over("user").alias("dt")])
           .group_by("user")
           .agg([
               pl.len().alias("n_events"),
               pl.col("dt").drop_nulls().median().alias("median_dt"),
               pl.col("dt").drop_nulls().mean().alias("mean_dt"),
               pl.col("dt").drop_nulls().std().alias("std_dt"),
               pl.col("dt").drop_nulls().quantile(0.90).alias("p90_dt"),
           ])
           .with_columns([
               pl.when(pl.col("n_events") < dt_min_events).then(0.0).otherwise(pl.col("median_dt")).alias("median_dt"),
               pl.when(pl.col("n_events") < dt_min_events).then(0.0).otherwise(pl.col("mean_dt")).alias("mean_dt"),
               pl.when(pl.col("n_events") < dt_min_events).then(0.0).otherwise(pl.col("std_dt")).alias("std_dt"),
               pl.when(pl.col("n_events") < dt_min_events).then(0.0).otherwise(pl.col("p90_dt")).alias("p90_dt"),
           ])
           .drop("n_events")
    )

    dest_share = (
        ldf.group_by(["user", "dst_computer"])
           .agg(pl.len().alias("cnt"))
           .group_by("user")
           .agg([
               pl.col("cnt").max().alias("top_dst_cnt"),
               pl.col("cnt").sum().alias("dst_total_cnt"),
               pl.col("cnt").mean().alias("dst_mean_cnt"),
               pl.col("cnt").std().alias("dst_std_cnt"),
           ])
           .with_columns([(pl.col("top_dst_cnt") / (pl.col("dst_total_cnt") + 1e-6)).alias("top_dst_share")])
    )

    logon_div = (
        ldf.group_by(["user", "logon_type"])
           .agg(pl.len().alias("lt_cnt"))
           .group_by("user")
           .agg([
               pl.col("logon_type").n_unique().alias("unique_logon_types"),
               pl.col("lt_cnt").max().alias("top_logon_cnt"),
               pl.col("lt_cnt").sum().alias("logon_total_cnt"),
           ])
           .with_columns([(pl.col("top_logon_cnt") / (pl.col("logon_total_cnt") + 1e-6)).alias("top_logon_share")])
    )

    feat = (
        core.join(gaps, on="user", how="left")
            .join(dest_share, on="user", how="left")
            .join(logon_div, on="user", how="left")
            .with_columns(pl.all().exclude("user").fill_null(0))
    )
    return feat

NO_RATIO_PREFIXES = ("median_dt", "mean_dt", "std_dt", "p90_dt")
NO_RATIO_EXACT = {"fail_rate", "fail_rate_2", "night_rate", "weekend_rate"}

def add_deviation_safe(df: pl.DataFrame, baseline: pl.DataFrame, base_feature_cols: list[str]) -> pl.DataFrame:
    out = df.join(baseline, on="user", how="left")

    eps = 1e-3

    for c in base_feature_cols:
        base_c = f"base_{c}"

        out = out.with_columns(
            (pl.col(c) - pl.col(base_c)).fill_null(0).alias(f"diff_{c}")
        )

        skip_ratio = (c in NO_RATIO_EXACT) or c.startswith(NO_RATIO_PREFIXES)
        if not skip_ratio:
            out = out.with_columns(
                (pl.max_horizontal(pl.col(base_c), pl.lit(eps)).log1p()).alias(f"_tmp_base_log_{c}")
            ).with_columns(
                (pl.max_horizontal(pl.col(c), pl.lit(0)).log1p()).alias(f"_tmp_cur_log_{c}")
            ).with_columns(
                (pl.col(f"_tmp_cur_log_{c}") - pl.col(f"_tmp_base_log_{c}"))
                .clip(-10, 10)
                .fill_null(0)
                .alias(f"log_ratio_{c}")
            ).drop([f"_tmp_base_log_{c}", f"_tmp_cur_log_{c}"])

    out = out.drop([f"base_{c}" for c in base_feature_cols])
    return out

def zscore_with_ref(x, ref):
    mu = ref.mean()
    sd = ref.std() + 1e-8
    return (x - mu) / sd

def topk_idx(scores, k):
    return set(np.argsort(scores)[::-1][:k])

# ----------------------------
# 1) Load + split
# ----------------------------
auth = pl.scan_parquet("filtered_auth.parquet")
times = auth.select(pl.col("time").cast(pl.Int64)).collect()
q70 = int(times["time"].quantile(0.70))
q85 = int(times["time"].quantile(0.85))
print("q70:", q70, "q85:", q85)

train_auth = auth.filter(pl.col("time") <= q70)
val_auth   = auth.filter((pl.col("time") > q70) & (pl.col("time") <= q85))
test_auth  = auth.filter(pl.col("time") > q85)

# ----------------------------
# 2) Feature tables (dt-stable)
# ----------------------------
train_df = user_features(train_auth, dt_min_events=30).collect()
val_df   = user_features(val_auth,   dt_min_events=30).collect()
test_df  = user_features(test_auth,  dt_min_events=30).collect()

print("Users train/val/test:", train_df.height, val_df.height, test_df.height)

# ----------------------------
# 3) TRAIN baselines + safe diff/log-ratio
# ----------------------------
base_feature_cols = [c for c in train_df.columns if c != "user"]
baseline = (
    train_df.select(["user"] + base_feature_cols)
            .group_by("user")
            .agg([pl.col(c).mean().alias(f"base_{c}") for c in base_feature_cols])
)

train_df2 = add_deviation_safe(train_df, baseline, base_feature_cols)
val_df2   = add_deviation_safe(val_df, baseline, base_feature_cols)
test_df2  = add_deviation_safe(test_df, baseline, base_feature_cols)

non_feature_cols = {"user", "label", "compromise_events", "src_user"}
feature_cols2 = [
    c for c, dtype in zip(train_df2.columns, train_df2.dtypes)
    if c not in non_feature_cols and str(dtype) in ("Float64","Float32","Int64","Int32","Int16","Int8")
]
print("All numeric features:", len(feature_cols2))

# ----------------------------
# 4) AE features: structure-only AND dt removed
# ----------------------------
drop_keywords = [
    "total_logins",
    "failed_logins",
    "_total_cnt",
    "_cnt",
    "top_dst_cnt",
    "top_logon_cnt",
]

dt_keys = ["median_dt", "mean_dt", "std_dt", "p90_dt"]

feature_cols_ae = [
    c for c in feature_cols2
    if not any(key in c for key in drop_keywords)
    and not any(k == c or c.endswith(k) or k in c for k in dt_keys)
]
print("AE features (no volume, no dt):", len(feature_cols_ae))

# ----------------------------
# 5) Matrices + scaling (AE)
# ----------------------------
X_train_ae = train_df2.select(feature_cols_ae).to_numpy()
X_val_ae   = val_df2.select(feature_cols_ae).to_numpy()
X_test_ae  = test_df2.select(feature_cols_ae).to_numpy()

scaler_ae = RobustScaler()
X_train_ae_s = scaler_ae.fit_transform(X_train_ae)
X_val_ae_s   = scaler_ae.transform(X_val_ae)
X_test_ae_s  = scaler_ae.transform(X_test_ae)

X_train_ae_s = np.clip(np.nan_to_num(X_train_ae_s), -15, 15)
X_val_ae_s   = np.clip(np.nan_to_num(X_val_ae_s),   -15, 15)
X_test_ae_s  = np.clip(np.nan_to_num(X_test_ae_s),  -15, 15)

input_dim = X_train_ae_s.shape[1]
print("AE input_dim:", input_dim)

# ----------------------------
# 6) Train AE (small sweep)
# ----------------------------
class NumpyDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

train_loader = DataLoader(NumpyDataset(X_train_ae_s), batch_size=4096, shuffle=True, num_workers=0)
val_tensor   = torch.tensor(X_val_ae_s, dtype=torch.float32)
test_tensor  = torch.tensor(X_test_ae_s, dtype=torch.float32)

class AutoEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=16, dropout=0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

def train_ae(train_loader, val_tensor, cfg, epochs=40, patience=5):
    model = AutoEncoder(input_dim, cfg["hidden_dim"], cfg["latent_dim"], cfg["dropout"]).to(device)
    opt = optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    bad = 0

    for _ in range(epochs):
        model.train()
        for xb in train_loader:
            xb = xb.to(device)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), xb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            xv = val_tensor.to(device)
            val_loss = loss_fn(model(xv), xv).item()

        if not np.isfinite(val_loss):
            val_loss = float("inf")

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val

def recon_scores_mae(model, X_tensor):
    model.eval()
    with torch.no_grad():
        x = X_tensor.to(device)
        r = model(x)
        return (r - x).abs().mean(dim=1).detach().cpu().numpy()

search_space = {
    "hidden_dim": [64, 128],
    "latent_dim": [8, 16],
    "dropout":    [0.0, 0.1, 0.2],
    "lr":         [3e-4, 1e-3],
    "weight_decay":[0.0, 1e-4],
}

best_model = None
best_cfg = None
best_val_loss = float("inf")

total = (
    len(search_space["hidden_dim"]) *
    len(search_space["latent_dim"]) *
    len(search_space["dropout"]) *
    len(search_space["lr"]) *
    len(search_space["weight_decay"])
)
trial = 0

for hd in search_space["hidden_dim"]:
    for ld in search_space["latent_dim"]:
        for dr in search_space["dropout"]:
            for lr in search_space["lr"]:
                for wd in search_space["weight_decay"]:
                    trial += 1
                    cfg = {"hidden_dim": hd, "latent_dim": ld, "dropout": dr, "lr": lr, "weight_decay": wd}
                    model, vloss = train_ae(train_loader, val_tensor, cfg)
                    if best_model is None or vloss < best_val_loss:
                        best_val_loss = vloss
                        best_model = model
                        best_cfg = cfg
                        print(f"[{trial}/{total}] NEW BEST val_mse={best_val_loss:.6f} cfg={best_cfg}")
                    elif trial % 10 == 0:
                        print(f"[{trial}/{total}] current best val_mse={best_val_loss:.6f}")

print("\nFINAL BEST AE CONFIG:", best_cfg)
print("FINAL BEST VAL MSE:", best_val_loss)

val_ae_score  = recon_scores_mae(best_model, val_tensor)
test_ae_score = recon_scores_mae(best_model, test_tensor)

# ----------------------------
# 7) Isolation Forest scoring (full features)
# ----------------------------
X_train_if = train_df2.select(feature_cols2).to_numpy()
X_val_if   = val_df2.select(feature_cols2).to_numpy()
X_test_if  = test_df2.select(feature_cols2).to_numpy()

scaler_if = RobustScaler()
X_train_if_s = scaler_if.fit_transform(X_train_if)
X_val_if_s   = scaler_if.transform(X_val_if)
X_test_if_s  = scaler_if.transform(X_test_if)

X_train_if_s = np.clip(np.nan_to_num(X_train_if_s), -15, 15)
X_val_if_s   = np.clip(np.nan_to_num(X_val_if_s),   -15, 15)
X_test_if_s  = np.clip(np.nan_to_num(X_test_if_s),  -15, 15)

iso = IsolationForest(n_estimators=500, contamination=0.01, random_state=42, n_jobs=-1)
iso.fit(X_train_if_s)

val_if_score  = -iso.decision_function(X_val_if_s)
test_if_score = -iso.decision_function(X_test_if_s)

# ----------------------------
# 8) Normalize on VAL + combine
# ----------------------------
test_if_z = zscore_with_ref(test_if_score, val_if_score)
test_ae_z = zscore_with_ref(test_ae_score, val_ae_score)

w_if = 0.7
w_ae = 0.3
test_combo = w_if * test_if_z + w_ae * test_ae_z

# diagnostics
corr = np.corrcoef(test_if_score, test_ae_score)[0, 1]
print("\nCorrelation(IF, AE) on TEST:", corr)
for k in [50, 100, 200]:
    ov = len(topk_idx(test_if_score, k) & topk_idx(test_ae_score, k))
    print(f"Top-{k} overlap:", ov, "/", k)

# ----------------------------
# 9) Alerts table + active-user filter
# ----------------------------
test_scored = test_df2.with_columns([
    pl.Series("if_score", test_if_score),
    pl.Series("ae_score", test_ae_score),
    pl.Series("combo_score", test_combo),
])

MIN_LOGINS = 50
test_scored = test_scored.filter(pl.col("total_logins") >= MIN_LOGINS)

cols_to_show = [
    "user", "combo_score", "if_score", "ae_score",
    "total_logins", "failed_logins", "fail_rate",
    "unique_destinations", "unique_src_computers",
    "dst_per_login", "src_per_login",
    "top_dst_share", "unique_logon_types", "top_logon_share",
    "median_dt", "std_dt"
]
cols_to_show = [c for c in cols_to_show if c in test_scored.columns]

print("\nTop 30 alerts by COMBO score (active users only):")
print(test_scored.sort("combo_score", descending=True).select(cols_to_show).head(30))

q70: 1853062 q85: 2234039
Users train/val/test: 11376 10018 10225
All numeric features: 55
AE features (no volume, no dt): 28
AE input_dim: 28
device: cpu
[1/48] NEW BEST val_mse=5.845150 cfg={'hidden_dim': 64, 'latent_dim': 8, 'dropout': 0.0, 'lr': 0.0003, 'weight_decay': 0.0}
[2/48] NEW BEST val_mse=4.900709 cfg={'hidden_dim': 64, 'latent_dim': 8, 'dropout': 0.0, 'lr': 0.0003, 'weight_decay': 0.0001}
[3/48] NEW BEST val_mse=4.443513 cfg={'hidden_dim': 64, 'latent_dim': 8, 'dropout': 0.0, 'lr': 0.001, 'weight_decay': 0.0}
[4/48] NEW BEST val_mse=4.352859 cfg={'hidden_dim': 64, 'latent_dim': 8, 'dropout': 0.0, 'lr': 0.001, 'weight_decay': 0.0001}
[8/48] NEW BEST val_mse=4.303908 cfg={'hidden_dim': 64, 'latent_dim': 8, 'dropout': 0.1, 'lr': 0.001, 'weight_decay': 0.0001}
[10/48] current best val_mse=4.303908
[20/48] current best val_mse=4.303908
[28/48] NEW BEST val_mse=4.263290 cfg={'hidden_dim': 128, 'latent_dim': 8, 'dropout': 0.0, 'lr': 0.001, 'weight_decay': 0.0001}
[30/48] current